# Commodity Data Retrieval and Pipeline

## Learning Objectives

By completing this notebook, you will:
1. Retrieve commodity futures prices from Yahoo Finance
2. Fetch fundamental data from public APIs
3. Handle missing values and align time series
4. Create a reproducible data pipeline

## Prerequisites
- Python environment with yfinance, pandas installed
- Internet connection for API access

## Estimated Time: 60 minutes

---

In [ ]:
learning_objectives(["Retrieve commodity futures prices from Yahoo Finance", "Fetch fundamental data from public APIs", "Handle missing values and align time series", "Create a reproducible data pipeline", "Python environment with yfinance, pandas installed", "Internet connection for API access"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)

print("Setup complete!")

## 1. Fetching Commodity Futures Prices

Yahoo Finance provides free access to continuous futures contracts. These are synthetic series that roll from one contract to the next.

**Important:** Continuous contracts may have price gaps at roll dates. For returns-based analysis, this is usually acceptable.

In [ ]:
# Define commodities to retrieve
COMMODITIES = {
    'CL=F': 'WTI Crude Oil',
    'NG=F': 'Natural Gas',
    'GC=F': 'Gold',
    'ZC=F': 'Corn',
    'ZS=F': 'Soybeans',
    'HG=F': 'Copper'
}

def fetch_commodity_prices(tickers, start_date='2015-01-01', end_date=None):
    """
    Fetch commodity futures prices from Yahoo Finance.
    
    Parameters
    ----------
    tickers : dict
        Dictionary mapping Yahoo ticker to commodity name
    start_date : str
        Start date for data retrieval
    end_date : str, optional
        End date (defaults to today)
    
    Returns
    -------
    pd.DataFrame
        DataFrame with adjusted close prices
    """
    if end_date is None:
        end_date = datetime.now().strftime('%Y-%m-%d')
    
    prices = {}
    
    for ticker, name in tickers.items():
        print(f"Fetching {name} ({ticker})...")
        try:
            data = yf.download(ticker, start=start_date, end=end_date, progress=False)
            if len(data) > 0:
                prices[name] = data['Adj Close']
                print(f"  Retrieved {len(data)} observations")
            else:
                print(f"  Warning: No data retrieved")
        except Exception as e:
            print(f"  Error: {e}")
    
    return pd.DataFrame(prices)

# Fetch data
prices_df = fetch_commodity_prices(COMMODITIES)
print(f"\nRetrieved data from {prices_df.index.min()} to {prices_df.index.max()}")

In [ ]:
# Examine the data
print("Price Data Summary:")
print(prices_df.describe())

# Check for missing values
print("\nMissing Values:")
print(prices_df.isnull().sum())

In [ ]:
# Visualize price history
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

for ax, col in zip(axes.flat, prices_df.columns):
    prices_df[col].plot(ax=ax, title=col)
    ax.set_ylabel('Price')

plt.tight_layout()
plt.show()

## 2. Computing Returns and Basic Features

For time series modeling, we typically work with returns rather than prices because:
1. Returns are (more) stationary
2. Returns are scale-invariant
3. Returns align with trading P&L

In [ ]:
def compute_returns_features(prices):
    """
    Compute return-based features from price data.
    
    Parameters
    ----------
    prices : pd.DataFrame
        DataFrame with price columns
    
    Returns
    -------
    pd.DataFrame
        DataFrame with original prices and computed features
    """
    features = prices.copy()
    
    for col in prices.columns:
        # Daily returns
        features[f'{col}_return'] = prices[col].pct_change()
        
        # Log returns (better for compounding)
        features[f'{col}_log_return'] = np.log(prices[col]).diff()
        
        # Realized volatility (20-day rolling)
        features[f'{col}_vol_20d'] = features[f'{col}_return'].rolling(20).std() * np.sqrt(252)
        
        # Momentum (20-day return)
        features[f'{col}_mom_20d'] = prices[col].pct_change(20)
        
        # Mean reversion indicator (z-score vs 60-day mean)
        rolling_mean = prices[col].rolling(60).mean()
        rolling_std = prices[col].rolling(60).std()
        features[f'{col}_zscore_60d'] = (prices[col] - rolling_mean) / rolling_std
    
    return features

# Compute features
features_df = compute_returns_features(prices_df)
print(f"Created {len(features_df.columns)} features")
print(features_df.columns.tolist())

In [ ]:
# Visualize WTI Crude Oil features
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

# Price
axes[0].plot(features_df['WTI Crude Oil'])
axes[0].set_ylabel('Price ($)')
axes[0].set_title('WTI Crude Oil')

# Returns
axes[1].plot(features_df['WTI Crude Oil_return'])
axes[1].set_ylabel('Daily Return')
axes[1].axhline(0, color='black', linewidth=0.5)

# Volatility
axes[2].plot(features_df['WTI Crude Oil_vol_20d'])
axes[2].set_ylabel('20d Volatility (ann.)')

# Z-score
axes[3].plot(features_df['WTI Crude Oil_zscore_60d'])
axes[3].set_ylabel('60d Z-Score')
axes[3].axhline(0, color='black', linewidth=0.5)
axes[3].axhline(2, color='red', linewidth=0.5, linestyle='--')
axes[3].axhline(-2, color='green', linewidth=0.5, linestyle='--')

plt.tight_layout()
plt.show()

## 3. Handling Missing Data

Commodity data often has gaps due to:
- Exchange holidays
- Data source outages
- Different trading calendars

We need robust strategies for handling these gaps.

In [ ]:
def analyze_missing_data(df):
    """
    Analyze patterns in missing data.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to analyze
    
    Returns
    -------
    pd.DataFrame
        Summary of missing data by column
    """
    summary = pd.DataFrame({
        'missing_count': df.isnull().sum(),
        'missing_pct': (df.isnull().sum() / len(df) * 100).round(2),
        'first_valid': df.apply(lambda x: x.first_valid_index()),
        'last_valid': df.apply(lambda x: x.last_valid_index())
    })
    return summary

# Analyze missing data in prices
missing_summary = analyze_missing_data(prices_df)
print("Missing Data Analysis:")
print(missing_summary)

In [ ]:
def handle_missing_prices(df, method='ffill', max_gap=5):
    """
    Handle missing values in price data.
    
    Parameters
    ----------
    df : pd.DataFrame
        Price DataFrame
    method : str
        'ffill' (forward fill) or 'interpolate'
    max_gap : int
        Maximum consecutive gaps to fill
    
    Returns
    -------
    pd.DataFrame
        DataFrame with missing values handled
    """
    df_clean = df.copy()
    
    if method == 'ffill':
        # Forward fill with limit
        df_clean = df_clean.ffill(limit=max_gap)
    elif method == 'interpolate':
        # Linear interpolation with limit
        df_clean = df_clean.interpolate(method='linear', limit=max_gap)
    
    # Report remaining gaps
    remaining = df_clean.isnull().sum().sum()
    print(f"Filled missing values. Remaining gaps: {remaining}")
    
    return df_clean

# Clean the price data
prices_clean = handle_missing_prices(prices_df, method='ffill', max_gap=3)

## 4. Calendar Features

Seasonality is crucial for commodity forecasting. Let's create calendar-based features.

In [ ]:
def add_calendar_features(df):
    """
    Add calendar-based features for seasonality modeling.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with DatetimeIndex
    
    Returns
    -------
    pd.DataFrame
        DataFrame with added calendar features
    """
    df = df.copy()
    
    # Basic calendar
    df['year'] = df.index.year
    df['month'] = df.index.month
    df['week'] = df.index.isocalendar().week
    df['dayofweek'] = df.index.dayofweek
    df['dayofyear'] = df.index.dayofyear
    
    # Seasonal indicators
    df['quarter'] = df.index.quarter
    df['is_month_start'] = df.index.is_month_start.astype(int)
    df['is_month_end'] = df.index.is_month_end.astype(int)
    
    # Fourier features for smooth seasonality
    day_of_year = df.index.dayofyear
    for k in [1, 2, 3]:  # Harmonics
        df[f'sin_{k}'] = np.sin(2 * np.pi * k * day_of_year / 365.25)
        df[f'cos_{k}'] = np.cos(2 * np.pi * k * day_of_year / 365.25)
    
    return df

# Add calendar features
prices_with_calendar = add_calendar_features(prices_clean)
print("Calendar features added:")
print([c for c in prices_with_calendar.columns if c not in prices_clean.columns])

In [ ]:
# Visualize seasonal patterns
seasonal_avg = prices_with_calendar.groupby('month')['Natural Gas'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
seasonal_avg.plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Month')
ax.set_ylabel('Average Price')
ax.set_title('Natural Gas Average Price by Month')
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.tight_layout()
plt.show()

---

## Exercise 1: Build a Complete Data Pipeline

**Task:** Create a function that:
1. Fetches price data for a given commodity
2. Handles missing values
3. Computes returns and volatility features
4. Adds calendar features
5. Returns a clean, feature-rich DataFrame

In [ ]:
# YOUR CODE HERE
def build_commodity_dataset(ticker, name, start_date='2015-01-01'):
    """
    Build a complete commodity dataset with features.
    
    Parameters
    ----------
    ticker : str
        Yahoo Finance ticker (e.g., 'CL=F')
    name : str
        Human-readable name
    start_date : str
        Start date for data
    
    Returns
    -------
    pd.DataFrame
        Complete dataset with price, returns, and calendar features
    """
    # Step 1: Fetch data
    # YOUR CODE
    
    # Step 2: Handle missing values
    # YOUR CODE
    
    # Step 3: Compute features
    # YOUR CODE
    
    # Step 4: Add calendar features
    # YOUR CODE
    
    return None  # Replace with your result

# Test your function
wti_data = build_commodity_dataset('CL=F', 'WTI')
if wti_data is not None:
    print(f"Dataset shape: {wti_data.shape}")
    print(f"Features: {wti_data.columns.tolist()}")

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_1():
    """Test the data pipeline function."""
    result = build_commodity_dataset('GC=F', 'Gold', '2020-01-01')
    
    assert result is not None, "Function returned None. Implement the pipeline!"
    assert isinstance(result, pd.DataFrame), "Result should be a DataFrame"
    assert len(result) > 100, "Should have at least 100 observations"
    assert 'Gold' in str(result.columns), "Should include price column"
    assert result.isnull().sum().sum() < len(result), "Should handle missing values"
    
    print("✅ Exercise 1 passed!")

test_exercise_1()

---

## Summary

### Key Takeaways
1. Yahoo Finance provides free commodity futures data via `yfinance`
2. Always check for and handle missing values appropriately
3. Returns-based features are often better than price-based for modeling
4. Calendar features capture seasonality patterns
5. A robust data pipeline is the foundation of any forecasting system

### What's Next
In the next notebook, we'll explore seasonality decomposition methods in depth.

### Additional Resources
- yfinance documentation: https://pypi.org/project/yfinance/
- EIA API guide in `guides/01_data_sources.md`